# Stages 1-4 End-to-End Test Runner

Walks a single query through flow routing (stage 1), query understanding (stage 2), stage-3 orchestration (translate + execute every endpoint with full visibility), and stage-4 reranking (pool scoring and top-K ranking). Unlike `run_stage_4`, this notebook captures every stage-3 LLM spec so they can be inspected alongside the per-endpoint execution results.

## Cell 1 — Setup, imports, DB readiness

Run once. Opens the Postgres pool, initializes Redis, and pings Qdrant (idempotent — safe to rerun). Every import used anywhere below lives here.

In [1]:
import sys, time, asyncio
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Iterable, Protocol

# Ensure project root is on the path so imports resolve.
project_root = str(Path.cwd().parent) if Path.cwd().name == "search_v2" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(Path(project_root) / ".env")

from implementation.llms.generic_methods import LLMProvider, generate_llm_response_async

# Stage 1 / 2
from search_v2.stage_1 import route_query
from search_v2.stage_2 import _STEP_2A_SYSTEM_PROMPT, _run_step_2b_for_concept

# Stage-4 building blocks (also drive stage 3 end-to-end with spec capture)
from search_v2.stage_4.flow_detection import detect_flow, tag_items
from search_v2.stage_4.dispatch import translate_item, execute_item
from search_v2.stage_4.assembly import assemble_pool, apply_deterministic_exclusions
from search_v2.stage_4.scoring import score_pool
from search_v2.stage_4.display import build_display_payload
from search_v2.stage_4.types import EndpointOutcome, Stage4Flow

# Shared schemas
from schemas.endpoint_result import EndpointResult
from schemas.enums import EndpointRoute, SearchFlow
from schemas.movie_input import load_movie_input_data
from schemas.query_understanding import QueryConcept, QueryUnderstandingResponse, Step2AResponse

# DB
from db.postgres import pool as postgres_pool, fetch_movie_cards
from db.qdrant import qdrant_client, check_qdrant
from db.redis import init_redis, check_redis
import db.redis as _redis_module

# DB readiness (idempotent — mirrors test_stage_3.ipynb)
if postgres_pool._closed:
    await postgres_pool.open()
    print("Postgres: pool opened")
else:
    print("Postgres: pool already open")

if _redis_module._redis_client is None:
    await init_redis()
    print(f"Redis:    initialized ({await check_redis()})")
else:
    print(f"Redis:    already initialized ({await check_redis()})")

print(f"Qdrant:   {await check_qdrant()}")


/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Postgres: pool opened
Redis:    initialized (ok)
Qdrant:   ok


## Cell 2 — Configuration

Set the query, LLM preset, and top-K. Uncomment one preset block or assign the three variables manually.

In [6]:
# ---- Query ----
query = "Scary movie"

# ---- Presets (uncomment one) ----
# Kimi K2.5 — no thinking
# provider, model, kwargs = LLMProvider.KIMI, "kimi-k2.5", {"enable_thinking": False}

# GPT 5.4 Mini — no thinking
provider, model, kwargs = LLMProvider.OPENAI, "gpt-5.4-mini", {"reasoning_effort": "none", "verbosity": "low"}

# Gemini 3 Flash — no thinking
# provider, model, kwargs = LLMProvider.GEMINI, "gemini-3-flash-preview", {"thinking_config": {"thinking_budget": 0}}

# ---- Shared ----
today = date.today()
TOP_K = 100

print(f"Query:    {query}")
print(f"Provider: {provider.value}")
print(f"Model:    {model}")
print(f"Kwargs:   {kwargs or '(defaults)'}")
print(f"Today:    {today.isoformat()}")
print(f"Top-K:    {TOP_K}")

Query:    Scary movie
Provider: openai
Model:    gpt-5.4-mini
Kwargs:   {'reasoning_effort': 'none', 'verbosity': 'low'}
Today:    2026-04-22
Top-K:    100


## Cell 3 — Stage 1: Flow Routing

In [7]:
start = time.perf_counter()
stage_1_response, stage_1_in_tok, stage_1_out_tok = await route_query(query)
stage_1_elapsed = time.perf_counter() - start

print(f"Stage 1 completed in {stage_1_elapsed:.2f}s")
print(f"Tokens — input: {stage_1_in_tok:,}  output: {stage_1_out_tok:,}\n")
print(stage_1_response.model_dump_json(indent=2))

Stage 1 completed in 2.69s
Tokens — input: 5,668  output: 257

{
  "ambiguity_analysis": "readings:\n- the movie titled \"Scary Movie\" -> primary\n- movies that are scary -> emit as alt (genre search returns an entirely different candidate set)",
  "query_traits": "traits: Scary Movie, scary, horror",
  "primary_intent": {
    "routing_signals": "The query exactly matches a well-known movie title.",
    "intent_rewrite": "The movie titled \"Scary Movie\"",
    "flow": "exact_title",
    "display_phrase": "Scary Movie",
    "title": "Scary Movie"
  },
  "alternative_intents": [
    {
      "routing_signals": "The phrase \"Scary movie\" is also a common natural-language way to ask for the horror genre.",
      "difference_rationale": "Switching from a specific title search to a genre search provides a broader discovery experience.",
      "intent_rewrite": "Movies that are scary",
      "flow": "standard",
      "display_phrase": "Scary movies",
      "title": null
    }
  ],
  "creativ

## Cell 4A — Stage 2A: Concept Extraction

Picks the first STANDARD-flow intent from stage 1, then runs the standalone Step 2A concept-inventory pass on its `intent_rewrite`. Non-standard flows (title lookup, similarity) are not supported by stages 3-4 in this notebook.


In [7]:
# Pick the first standard-flow intent. Change this filter or index to test a different one.
stage_1_intents = [stage_1_response.primary_intent, *stage_1_response.alternative_intents]
selected_intent = next(
    (intent for intent in stage_1_intents if intent.flow == SearchFlow.STANDARD),
    None,
)
if selected_intent is None:
    raise RuntimeError(
        "No standard-flow intent in stage 1 output — this notebook "
        "only drives the standard pipeline. Tweak the query."
    )

intent_rewrite = selected_intent.intent_rewrite
print(f"Selected intent: {selected_intent.display_phrase}")
print(f"Intent rewrite:          {intent_rewrite}\n")

start = time.perf_counter()
step_2a_response, stage_2a_in_tok, stage_2a_out_tok = await generate_llm_response_async(
    provider=provider,
    user_prompt=f"Interpretation rewrite:\n{intent_rewrite}",
    system_prompt=_STEP_2A_SYSTEM_PROMPT,
    response_format=Step2AResponse,
    model=model,
    **kwargs,
)
stage_2a_elapsed = time.perf_counter() - start

print(f"Stage 2A completed in {stage_2a_elapsed:.2f}s")
print(f"Tokens — input: {stage_2a_in_tok:,}  output: {stage_2a_out_tok:,}\n")
print("Ingredient inventory:")
for ingredient in step_2a_response.ingredient_inventory:
    print(f"- {ingredient}")
if step_2a_response.ingredient_inventory:
    print()
for extracted_concept in step_2a_response.concepts:
    print(f"- {extracted_concept.concept}")
    print(f"  Boundary note: {extracted_concept.boundary_note}")
    print(f"  Required ingredients: {', '.join(extracted_concept.required_ingredients)}")
if step_2a_response.concepts:
    print()
print(step_2a_response.model_dump_json(indent=2))


Selected intent: Joker Series
Intent rewrite:          Movies in the Joker film series

Stage 2A completed in 2.61s
Tokens — input: 753  output: 109

Ingredient inventory:
- Movies
- Joker film series

- Joker film series movies
  Boundary note: One entity/franchise concept: the user wants films belonging to the Joker film series.
  Required ingredients: Movies, Joker film series

{
  "ingredient_inventory": [
    "Movies",
    "Joker film series"
  ],
  "concept_inventory_analysis": "This query contains one scope anchor: the Joker film series. 'Movies' is just the medium/type constraint for items within that series, not a separate user intent. So the concept stays unified.",
  "concepts": [
    {
      "boundary_note": "One entity/franchise concept: the user wants films belonging to the Joker film series.",
      "concept": "Joker film series movies",
      "required_ingredients": [
        "Movies",
        "Joker film series"
      ]
    }
  ]
}


## Cell 4B — Stage 2B: Per-Concept Expression Planning

Runs one Step 2B call per Step 2A concept, mirrors the production failure policy, and assembles the final `QueryUnderstandingResponse` that downstream stage 3/4 cells consume.


In [8]:
step_2b_input_tokens = 0
step_2b_output_tokens = 0
successful_concepts = []
dropped_concepts: list[tuple[str, Exception]] = []

if step_2a_response.concepts:
    start = time.perf_counter()
    raw_results = await asyncio.gather(
        *(
            _run_step_2b_for_concept(
                query=intent_rewrite,
                concept=extracted_concept,
                inventory=step_2a_response.concepts,
                provider=provider,
                model=model,
                **kwargs,
            )
            for extracted_concept in step_2a_response.concepts
        ),
        return_exceptions=True,
    )
    stage_2b_elapsed = time.perf_counter() - start

    for extracted_concept, raw_result in zip(step_2a_response.concepts, raw_results):
        if isinstance(raw_result, Exception):
            dropped_concepts.append((extracted_concept.concept, raw_result))
            continue

        step_2b_response, concept_in_tok, concept_out_tok = raw_result
        successful_concepts.append(
            QueryConcept(
                boundary_note=extracted_concept.boundary_note,
                concept=extracted_concept.concept,
                required_ingredients=extracted_concept.required_ingredients,
                expression_plan_analysis=step_2b_response.expression_plan_analysis,
                expressions=step_2b_response.expressions,
            )
        )
        step_2b_input_tokens += concept_in_tok
        step_2b_output_tokens += concept_out_tok
else:
    stage_2b_elapsed = 0.0

if step_2a_response.concepts and not successful_concepts:
    raise RuntimeError(
        "Step 2A extracted concepts, but every Step 2B concept was dropped."
    )

qu = QueryUnderstandingResponse(
    ingredient_inventory=step_2a_response.ingredient_inventory,
    concept_inventory_analysis=step_2a_response.concept_inventory_analysis,
    concepts=successful_concepts,
)
stage_2_in_tok = stage_2a_in_tok + step_2b_input_tokens
stage_2_out_tok = stage_2a_out_tok + step_2b_output_tokens

print(f"Stage 2B completed in {stage_2b_elapsed:.2f}s")
print(f"Tokens — input: {step_2b_input_tokens:,}  output: {step_2b_output_tokens:,}\n")

if dropped_concepts:
    print("Dropped concepts:")
    for concept_name, err in dropped_concepts:
        print(f"  - {concept_name}: {err}")
    print()

for concept in successful_concepts:
    print("=" * 72)
    print(f"CONCEPT: {concept.concept}")
    print("=" * 72)
    print(concept.model_dump_json(indent=2))
    print()

print(f"Assembled Stage 2 completed in {stage_2a_elapsed + stage_2b_elapsed:.2f}s")
print(f"Tokens — input: {stage_2_in_tok:,}  output: {stage_2_out_tok:,}\n")
print(qu.model_dump_json(indent=2))


Stage 2B completed in 2.75s
Tokens — input: 1,614  output: 149

CONCEPT: Joker film series movies
{
  "boundary_note": "One entity/franchise concept: the user wants films belonging to the Joker film series.",
  "concept": "Joker film series movies",
  "required_ingredients": [
    "Movies",
    "Joker film series"
  ],
  "expression_plan_analysis": "This concept needs both required ingredients preserved: the films must be movies, and they must belong to the Joker film series. One expression can cover both together via franchise_structure, since franchise membership naturally applies to the film set while keeping the movie boundary implicit; no second expression is needed.",
  "expressions": [
    {
      "coverage_ingredients": [
        "Movies",
        "Joker film series"
      ],
      "routing_rationale": "Franchise membership directly captures films belonging to the Joker film series, and this concept is specifically about the movie entries in that series.",
      "route": "franc

## Cell 5 — Stage 3 Orchestration

Translates every tagged item, executes the pool-independent ones unrestricted, then assembles the candidate pool and applies deterministic exclusions. Prints the LLM spec and execution result for each endpoint, followed by a consolidated stats block.

Pool-dependent items (inclusion dealbreakers / preferences that do not generate candidates, plus semantic exclusions) have their LLM specs printed here but their execution is deferred to stage 4 since it needs the assembled pool.


In [9]:
class _Scored(Protocol):
    movie_id: int
    score: float


async def print_scored_candidates(
    candidates: Iterable[_Scored],
    *,
    limit: int | None = 10,
    sort_desc: bool = True,
) -> None:
    """Print scored movies as `score  title (year)  [tmdb_id]`."""
    items_list = list(candidates)
    if not items_list:
        print("    (no scored candidates)")
        return
    if sort_desc:
        items_list = sorted(items_list, key=lambda c: c.score, reverse=True)
    if limit is not None:
        items_list = items_list[:limit]
    cards = await fetch_movie_cards([c.movie_id for c in items_list])
    by_id = {c["movie_id"]: c for c in cards}
    for cand in items_list:
        card = by_id.get(cand.movie_id)
        if card is None:
            title, year = "<missing card>", "?"
        else:
            title = card["title"] or "<untitled>"
            ts = card["release_ts"]
            year = (
                datetime.fromtimestamp(ts, tz=timezone.utc).year
                if ts is not None else "?"
            )
        print(f"    {cand.score:.3f}  {title} ({year})  [tmdb_id={cand.movie_id}]")


# Phase 0 — flow detect + item tagging -------------------------------------
flow = detect_flow(qu)
items = tag_items(qu, flow)

print(f"Stage-4 flow: {flow.value}")
if flow == Stage4Flow.BROWSE:
    raise RuntimeError(
        "BROWSE flow not supported by this notebook — the seed would come "
        "from movie_card popularity ordering; extend with fetch_browse_seed_ids."
    )

print(f"Tagged items ({len(items)}):")
for it in items:
    print(
        f"  {it.debug_key:32s}  endpoint={it.endpoint.value:22s}  "
        f"role={it.role:22s}  generates_candidates={it.generates_candidates}"
    )

# Partition items the same way run_stage_4 does.
candidate_generators = [i for i in items if i.generates_candidates]
det_exclusions = [
    i for i in items
    if i.role == "exclusion_dealbreaker" and i.endpoint != EndpointRoute.SEMANTIC
]
sem_exclusions = [
    i for i in items
    if i.role == "exclusion_dealbreaker" and i.endpoint == EndpointRoute.SEMANTIC
]
pool_dependent_items = (
    [i for i in items if i.role == "inclusion_dealbreaker" and not i.generates_candidates]
    + [i for i in items if i.role == "preference" and not i.generates_candidates]
    + sem_exclusions
)
pool_independent_items = candidate_generators + det_exclusions

# Phase 1 — translate every item in parallel (captures specs for stage 4).
translate_kwargs = dict(
    intent_rewrite=intent_rewrite,
    today=today,
    provider=provider,
    model=model,
)

start = time.perf_counter()
translate_results = await asyncio.gather(
    *(translate_item(i, **translate_kwargs) for i in items)
)
translate_elapsed = time.perf_counter() - start

specs_by_key: dict[str, tuple] = {
    it.debug_key: res for it, res in zip(items, translate_results)
}
print(f"\nAll {len(items)} translations completed in {translate_elapsed:.2f}s")

# Phase 2 — execute pool-independent items unrestricted in parallel.
start = time.perf_counter()
execute_results = await asyncio.gather(
    *(
        execute_item(it, specs_by_key[it.debug_key][0], None, qdrant_client=qdrant_client)
        for it in pool_independent_items
    )
)
execute_elapsed = time.perf_counter() - start

pool_independent_outcomes: list[EndpointOutcome] = []
for it, (result, exec_ms, exec_status, exec_err) in zip(pool_independent_items, execute_results):
    spec, llm_ms, translate_status, translate_err = specs_by_key[it.debug_key]
    # Mirror run_stage_4: if translation failed, surface the translation status.
    if translate_status != "ok":
        pool_independent_outcomes.append(EndpointOutcome(
            item=it, result=EndpointResult(), status=translate_status,
            llm_ms=llm_ms, exec_ms=None, error_message=translate_err,
        ))
    else:
        pool_independent_outcomes.append(EndpointOutcome(
            item=it, result=result, status=exec_status,
            llm_ms=llm_ms, exec_ms=exec_ms, error_message=exec_err,
        ))

print(
    f"All {len(pool_independent_items)} pool-independent executions "
    f"completed in {execute_elapsed:.2f}s\n"
)

# Per-endpoint printing.
print("=" * 72)
print("STAGE 3 PER-ENDPOINT OUTPUT")
print("=" * 72)

outcomes_by_key = {o.item.debug_key: o for o in pool_independent_outcomes}
pool_independent_keys = {it.debug_key for it in pool_independent_items}

for it in items:
    spec, llm_ms, translate_status, translate_err = specs_by_key[it.debug_key]
    print()
    print("-" * 72)
    header_llm = f"{llm_ms:.0f}ms" if llm_ms is not None else "n/a"
    print(
        f"[{it.debug_key}]  endpoint={it.endpoint.value}  role={it.role}  "
        f"LLM={header_llm}  translate_status={translate_status}"
    )
    print("-" * 72)
    print("LLM SPEC:")
    if translate_status != "ok":
        print(f"  translation {translate_status}: {translate_err}")
    elif spec is None:
        print("  (trending: no LLM translator)")
    else:
        print(spec.model_dump_json(indent=2))

    if it.debug_key in pool_independent_keys:
        outcome = outcomes_by_key[it.debug_key]
        ms = f"{outcome.exec_ms:.0f}ms" if outcome.exec_ms is not None else "n/a"
        print(
            f"\nEXECUTION — {len(outcome.result.scores)} matches  "
            f"exec={ms}  status={outcome.status}"
        )
        await print_scored_candidates(outcome.result.scores, limit=10)
    else:
        print("\nEXECUTION — deferred to stage 4 (pool-dependent)")

# Phases 3 + 4 — pool assembly + deterministic exclusion subtraction.
candidate_outcomes = [o for o in pool_independent_outcomes if o.item.generates_candidates]
det_exclusion_outcomes = [
    o for o in pool_independent_outcomes if o.item.role == "exclusion_dealbreaker"
]

pool = assemble_pool(candidate_outcomes, browse_seed_ids=None)
pool_after_generation = len(pool)
pool = apply_deterministic_exclusions(pool, det_exclusion_outcomes)
pool_after_exclusion = len(pool)

# Consolidated stats.
print()
print("=" * 72)
print("STAGE 3 CONSOLIDATED CANDIDATE STATS")
print("=" * 72)

print(f"\nCandidate generators ({len(candidate_outcomes)}):")
for o in candidate_outcomes:
    print(
        f"  {o.item.debug_key:32s}  endpoint={o.item.endpoint.value:22s}  "
        f"role={o.item.role:22s}  matched={len(o.result.scores)}"
    )

print(f"\nDeterministic exclusions ({len(det_exclusion_outcomes)}):")
for o in det_exclusion_outcomes:
    print(
        f"  {o.item.debug_key:32s}  endpoint={o.item.endpoint.value:22s}  "
        f"excluded_ids={len(o.result.scores)}"
    )

print()
print(f"Unioned inclusion IDs (pre-exclusions):  {pool_after_generation}")
print(f"Final candidate pool size:               {pool_after_exclusion}")

Stage-4 flow: standard
Tagged items (1):
  concept[0].expression[0]          endpoint=franchise_structure     role=inclusion_dealbreaker   generates_candidates=True

All 1 translations completed in 2.18s
All 1 pool-independent executions completed in 0.49s

STAGE 3 PER-ENDPOINT OUTPUT

------------------------------------------------------------------------
[concept[0].expression[0]]  endpoint=franchise_structure  role=inclusion_dealbreaker  LLM=2176ms  translate_status=ok
------------------------------------------------------------------------
LLM SPEC:
{
  "concept_analysis": "Specific lineage inside a named franchise. Signal phrase: \"Joker film series\" maps to franchise_or_universe_names; no named subgroup, no sequel/prequel/remake/reboot wording, no spinoff/crossover wording, and no launch behavior is requested.",
  "franchise_or_universe_names": [
    "joker"
  ],
  "recognized_subgroups": null,
  "lineage_position": null,
  "structural_flags": null,
  "launch_scope": null
}

EX

## Cell 6 — Stage 4 Reranking

Runs pool-dependent items against the assembled pool, fetches movie cards, scores with `score_pool`, sorts, and slices the top-K. You can set:

- `inspect_tmdb_ids` — any TMDB IDs you want a full per-endpoint breakdown for (works for any pool member, not just the top 100).
- `SHOW_OVERVIEWS` — flip to True to also print the overview text of the top-25 movies (fetched from the ingestion tracker DB).


In [10]:
# ---- User inputs ----
# TMDB IDs to drill into. Works for any id that made it through stage 3 exclusions;
# if an id was excluded at stage 3 it prints a clear message.
inspect_tmdb_ids: list[int] = [1726]

# Flip to True to print overview text for each of the top-25 movies.
SHOW_OVERVIEWS = False

TOP_N_SHOW = 25

if not pool:
    raise RuntimeError("Pool is empty — cannot run stage 4. Check stage 3 output.")

pool_set = set(pool)

async def _run_pool_dependent(it):
    spec, llm_ms, translate_status, translate_err = specs_by_key[it.debug_key]
    if translate_status != "ok":
        return EndpointOutcome(
            item=it, result=EndpointResult(), status=translate_status,
            llm_ms=llm_ms, exec_ms=None, error_message=translate_err,
        )
    result, exec_ms, exec_status, exec_err = await execute_item(
        it, spec, pool_set, qdrant_client=qdrant_client,
    )
    return EndpointOutcome(
        item=it, result=result, status=exec_status,
        llm_ms=llm_ms, exec_ms=exec_ms, error_message=exec_err,
    )

# Pool-dependent executions run first (mirrors run_stage_4 phase 5).
start = time.perf_counter()
pool_dep_coros = [_run_pool_dependent(it) for it in pool_dependent_items]
pool_dependent_outcomes = await asyncio.gather(*pool_dep_coros)
stage_4_exec_elapsed = time.perf_counter() - start

print(
    f"Pool-dependent executions completed in "
    f"{stage_4_exec_elapsed:.2f}s  (pool={len(pool)}, "
    f"pool-dep items={len(pool_dependent_items)})"
)

# Phase 6 — score composition.
all_outcomes = list(pool_independent_outcomes) + list(pool_dependent_outcomes)
inclusion_outcomes = [o for o in all_outcomes if o.item.role == "inclusion_dealbreaker"]
preference_outcomes = [o for o in all_outcomes if o.item.role == "preference"]
semantic_exclusion_outcomes = [
    o for o in all_outcomes
    if o.item.role == "exclusion_dealbreaker" and o.item.endpoint == EndpointRoute.SEMANTIC
]

breakdowns = score_pool(
    pool,
    inclusion_outcomes=inclusion_outcomes,
    preference_outcomes=preference_outcomes,
    semantic_exclusion_outcomes=semantic_exclusion_outcomes,
)

# Phase 7 — sort, slice, shape.
breakdowns_sorted = sorted(breakdowns, key=lambda b: b.final_score, reverse=True)
top = breakdowns_sorted[:TOP_K]
top_ids = [b.movie_id for b in top]
breakdown_by_id = {b.movie_id: b for b in breakdowns}
cards = await fetch_movie_cards(top_ids)
cards_by_id = {c["movie_id"]: c for c in cards}
movies = build_display_payload(top_ids, cards_by_id)
top_25 = movies[:TOP_N_SHOW]

# debug_key → endpoint lookup for readability in the breakdown block.
endpoint_by_key = {it.debug_key: it.endpoint.value for it in items}

# Per-endpoint breakdowns for inspected TMDB IDs.
print()
print("=" * 72)
print("PER-ENDPOINT BREAKDOWNS FOR inspect_tmdb_ids")
print("=" * 72)

if not inspect_tmdb_ids:
    print("\n(no ids supplied — set `inspect_tmdb_ids = [...]` to drill in)")
else:
    for tmdb_id in inspect_tmdb_ids:
        print()
        card = cards_by_id.get(tmdb_id)
        title = card["title"] if card else "<card unavailable>"
        b = breakdown_by_id.get(tmdb_id)
        if b is None:
            print(f"  {tmdb_id}: {title} — not in scored pool (excluded at stage 3)")
            continue
        print(f"  {tmdb_id}: {title}")
        print(f"    final_score             = {b.final_score:.4f}")
        print(f"    dealbreaker_sum         = {b.dealbreaker_sum:.4f}")
        print(f"    preference_contribution = {b.preference_contribution:.4f}")
        print(f"    exclusion_penalties     = {b.exclusion_penalties:.4f}")
        if b.per_item_scores:
            print("    per-endpoint contributions:")
            for debug_key, score in b.per_item_scores.items():
                ep = endpoint_by_key.get(debug_key, "prior")
                print(f"      {debug_key:32s}  endpoint={ep:22s}  score={score:.4f}")
        else:
            print("    (no endpoint contributed a non-zero score)")

# First 25 of top 100.
print()
print("=" * 72)
print(f"TOP {TOP_N_SHOW} OF TOP {TOP_K}")
print("=" * 72)
for rank, (movie, b) in enumerate(zip(top_25, top[:TOP_N_SHOW]), start=1):
    title = movie["movie_title"] or "<untitled>"
    release_date = movie["release_date"] or ""
    year = release_date[:4] if release_date else "?"
    print(
        f"  #{rank:<3d}  score={b.final_score:.4f}  "
        f"{title} ({year})  tmdb_id={movie['tmdb_id']}"
    )

# Optional overviews.
if SHOW_OVERVIEWS and top_25:
    print()
    print("=" * 72)
    print(f"OVERVIEWS FOR TOP {TOP_N_SHOW}")
    print("=" * 72)
    overview_data = load_movie_input_data([m["tmdb_id"] for m in top_25])
    for rank, movie in enumerate(top_25, start=1):
        tmdb_id = movie["tmdb_id"]
        title = movie["movie_title"] or "<untitled>"
        data = overview_data.get(tmdb_id)
        overview = (data.overview if data else "") or "(overview unavailable)"
        print()
        print(f"  #{rank}  {title}  [tmdb_id={tmdb_id}]")
        print(f"  {overview}")

Pool-dependent executions completed in 0.00s  (pool=1, pool-dep items=0)

PER-ENDPOINT BREAKDOWNS FOR inspect_tmdb_ids

  1726: <card unavailable> — not in scored pool (excluded at stage 3)

TOP 25 OF TOP 100
  #1    score=1.0000  Joker: Folie à Deux (2024)  tmdb_id=889737
